In [ ]:
import numpy as np

from vtkmodules.util.numpy_support import numpy_to_vtk
from vtkmodules.util import colors as color
from vtkmodules.util.data_model import (
    UnstructuredGrid,
    Points,
    CellArray
)

from vtkmodules.vtkCommonDataModel import (
    VTK_HEXAHEDRON
)

from vtkmodules.vtkRenderingCore import (
    vtkActor,
    vtkDataSetMapper,
    vtkRenderer,
    vtkRenderWindow
)

from trame.app import get_server
from trame.ui.vuetify3 import VAppLayout
from trame.widgets import vtk as vtk_widgets


In [ ]:
# Define the 8 corner points of a unit hexahedron
# Bottom face (z=0): nodes 0,1,2,3
# Top face (z=1): nodes 4,5,6,7
coordinates = np.array([
    [0.0, 0.0, 0.0],  # node 0 - bottom-left-front
    [1.0, 0.0, 0.0],  # node 1 - bottom-right-front
    [1.0, 1.0, 0.0],  # node 2 - bottom-right-back
    [0.0, 1.0, 0.0],  # node 3 - bottom-left-back
    [0.0, 0.0, 1.0],  # node 4 - top-left-front
    [1.0, 0.0, 1.0],  # node 5 - top-right-front
    [1.0, 1.0, 1.0],  # node 6 - top-right-back
    [0.0, 1.0, 1.0],  # node 7 - top-left-back
    [1.0, 0.0, 0.0],  # node 0 - bottom-left-front
    [2.0, 0.0, 0.0],  # node 1 - bottom-right-front
    [2.0, 1.0, 0.0],  # node 2 - bottom-right-back
    [1.0, 1.0, 0.0],  # node 3 - bottom-left-back
    [1.0, 0.0, 1.0],  # node 4 - top-left-front
    [2.0, 0.0, 1.0],  # node 5 - top-right-front
    [2.0, 1.0, 1.0],  # node 6 - top-right-back
    [1.0, 1.0, 1.0],  # node 7 - top-left-back
    [2.0, 0.0, 0.0],  # node 0 - bottom-left-front
    [3.0, 0.0, 0.0],  # node 1 - bottom-right-front
    [3.0, 1.0, 0.0],  # node 2 - bottom-right-back
    [2.0, 1.0, 0.0],  # node 3 - bottom-left-back
    [2.0, 0.0, 1.0],  # node 4 - top-left-front
    [3.0, 0.0, 1.0],  # node 5 - top-right-front
    [3.0, 1.0, 1.0],  # node 6 - top-right-back
    [2.0, 1.0, 1.0],  # node 7 - top-left-back
], dtype=np.float64)

points = Points(data=coordinates)

# ============================================================
# Create vtkCellArray
# ============================================================

# Connectivity array: node indices of the hexahedron
connectivity = np.arange(24, dtype=np.int64)

cell_types = np.array(
    [
        [VTK_HEXAHEDRON],
        [VTK_HEXAHEDRON],
        [VTK_HEXAHEDRON]
    ], dtype=np.uint8)

vtk_cell_types = numpy_to_vtk(cell_types, deep=True)

offsets = np.array([0, 8, 16, 24], dtype=np.int64)

cells = CellArray(offsets=offsets, connectivity=connectivity)

# ============================================================
# Create vtkUnstructuredGrid
# ============================================================

grid = UnstructuredGrid(points=points, cells=(vtk_cell_types, cells))


In [ ]:
# -------------------------
# VTK rendering pipeline
# -------------------------

mapper = vtkDataSetMapper(input_data=grid)

actor = vtkActor(mapper=mapper)
#actor.property.representation = VTK_WIREFRAME
actor.property.edge_visibility = True
actor.property.edge_color = (0.1, 0.1, 0.1)
actor.property.line_width = 1

renderer = vtkRenderer()
renderer.AddActor(actor)
renderer.ResetCamera()


render_window = vtkRenderWindow()
render_window.AddRenderer(renderer)


In [ ]:

# -------------------------
# Trame
# -------------------------

server = get_server()

with VAppLayout(server) as layout:
    #layout.title.set_text("VTK 4-Node Quad")
    view = vtk_widgets.VtkLocalView(render_window)

await layout.display_cell(height="600px")